# Hatrie SQL Reproducible Analysis

This notebook streams a Hatrie SQL query and persists the exact result as Parquet.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import requests

BASE_URL = os.environ.get('HATRIE_URL', 'http://127.0.0.1:8080').rstrip('/')
TOKEN = os.environ['HATRIE_CACHE_AUTH_TOKEN']
OUTPUT = Path('hatrie_query_result.parquet')


In [ ]:
def stream_sql(query):
    response = requests.post(
        f'{BASE_URL}/api/sql',
        headers={'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/x-ndjson'},
        json={'query': query, 'stream': True},
        stream=True,
        timeout=(5, 60),
    )
    response.raise_for_status()
    columns, rows = None, []
    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        message = json.loads(line)
        if message['type'] == 'columns':
            columns = message['columns']
        elif message['type'] == 'row':
            rows.append(message['row'])
        elif message['type'] == 'error':
            raise RuntimeError(message['error'])
    return pd.DataFrame(rows, columns=columns)


In [ ]:
query = "FROM KEYS AS key SELECT key LIMIT 1000"
result = stream_sql(query)
result.to_parquet(OUTPUT, engine='pyarrow', index=False)
result.head()
